# 对话缓冲记忆 (Conversation Buffer Memory)

> **最基本的 Agent 记忆形式：逐字存储整个对话。**

每个 AI Agent 都需要记忆。没有它，每轮对话都是一张白纸。Agent 无法引用你一分钟前说的话，更不用说维持连贯的多轮对话了。

想象一台永不暂停的录音机。它捕获每一个字，当 Agent 需要回复时，回放整盘磁带。这就是**对话缓冲记忆**。它是所有其他记忆技术构建的基础。思路很直接：

1. 保持一个有序的消息列表。
2. 每轮将*完整*列表发送给 LLM。
3. 追加回复。重复。

本 notebook 展示如何使用 **LangChain** 从零构建它。你还将对比 DIY 版本和 **LangChain 内置的 ConversationBufferMemory**。

**学完后你将理解：**
- 为什么缓冲记忆是任何 Agent 的默认起点。
- 消息如何累积，为什么成本和延迟每轮线性增长。
- 这种方法何时失效以及下一步该怎么做。

## 核心概念

- **消息列表 (Message list)**：一个有序的 `{role, content}` 字典列表。角色通常是 `user` 和 `assistant`（加上可选的 `system` 提示词）。
- **角色交替 (Role alternation)**：LLM 依赖正确的角色顺序来知道谁说了什么。
- **上下文注入 (Context injection)**：完整的消息列表在每次 API 调用时作为提示词传递。“上下文”指模型在回答前读取的文本。
- **Token 计数 (Token counting)**：每条消息消耗 token（模型内部使用的词片段）。累计总数决定成本以及是否会触及上下文窗口上限。
- **线性增长 (Linear growth)**：每轮新增 token。*所有*之前的 token 也会被重新发送。因此累计 token 使用量跨轮次呈 **O(n²)** 增长，其中 *n* 是轮次数。这意味着轮次翻倍，总成本大约翻四倍。

## 模型准备

In [1]:
# 导入Langchain的初始化模型的函数
from langchain.chat_models import init_chat_model
# 加载环境变量
from dotenv import load_dotenv
load_dotenv()

# 调用init_chat_model函数初始化模型，参数model用来指定模型名称，Langchain会根据模型名字自动设定base_url，并从环境变量中获取api_key
model = init_chat_model(model="deepseek-chat")
print(type(model)) # <class 'langchain_deepseek.chat_models.ChatDeepSeek'>

/Users/huanglu/Project/ai-agent-notes/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<class 'langchain_deepseek.chat_models.ChatDeepSeek'>


## 实现

我们将构建一个最小的 `ConversationBufferMemory` 类：
1. 用普通 Python 列表存储消息。
2. 暴露 `add_user_message` / `add_assistant_message` 辅助方法。
3. 每次调用将完整历史发送给 LLM。
4. 跟踪每轮 token 使用量以便后续可视化。

In [16]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

class ConversationBufferMemory:
    """基于 LangChain 的最小对话缓冲记忆。"""

    def __init__(
        self,
        model,  # LangChain ChatModel 实例
        system_prompt: str | None = None,
    ):
        self.model = model
        self.system_prompt = system_prompt

        # 核心数据结构：消息缓冲区
        self.messages: list[dict] = []

        # 用于 token 增长演示的记录
        self.turn_token_usage: list[dict] = []

    # ── 消息辅助方法 ──────────────────────────────────────────────
    def add_user_message(self, content: str) -> None:
        self.messages.append({"role": "user", "content": content})

    def add_assistant_message(self, content: str) -> None:
        self.messages.append({"role": "assistant", "content": content})

    # ── 对话 ─────────────────────────────────────────────────────────
    def _to_langchain_messages(self) -> list:
        """将内部 dict 列表转为 LangChain 消息对象。"""
        lc_messages = []
        if self.system_prompt:
            lc_messages.append(SystemMessage(content=self.system_prompt))
        for msg in self.messages:
            if msg["role"] == "user":
                lc_messages.append(HumanMessage(content=msg["content"]))
            elif msg["role"] == "assistant":
                lc_messages.append(AIMessage(content=msg["content"]))
        return lc_messages

    def chat(self, user_input: str) -> str:
        """发送消息并获取回复。每次发送*完整*缓冲区。"""
        self.add_user_message(user_input)

        lc_messages = self._to_langchain_messages()
        response = self.model.invoke(lc_messages)

        assistant_text = response.content
        self.add_assistant_message(assistant_text)

        # 记录本轮 token 使用量
        usage = response.usage_metadata or {}
        self.turn_token_usage.append({
            "turn": len(self.turn_token_usage) + 1,
            "input_tokens": usage.get("input_tokens", 0),
            "output_tokens": usage.get("output_tokens", 0),
        })

        return assistant_text

    # ── 工具方法 ────────────────────────────────────────────────────
    def get_history(self) -> list[dict]:
        import copy
        return copy.deepcopy(self.messages)

    def clear(self) -> None:
        self.messages.clear()
        self.turn_token_usage.clear()

    def __len__(self) -> int:
        return len(self.messages)

    def __repr__(self) -> str:
        return f"ConversationBufferMemory({len(self.messages)} messages)"

## 运行示例

一个简短的多轮对话展示缓冲记忆的实际效果。Agent 记住了一切，因为它每轮都重新读取完整历史。

In [18]:
memory = ConversationBufferMemory(
    model=model,
    system_prompt="You are a helpful, concise assistant. Keep replies under 2 sentences.",
)

# 多轮对话
exchanges = [
    "Hi! My name is Alice and I'm a machine-learning engineer.",
    "I'm working on a project about agent memory. Any tips?",
    "What's my name and what do I do?",  # 回忆测试
]

for msg in exchanges:
    print(f"👤 User:  {msg}")
    reply = memory.chat(msg)
    print(f"🤖 Agent: {reply}\n")

print(f"缓冲区现在包含 {len(memory)} 条消息。")

👤 User:  Hi! My name is Alice and I'm a machine-learning engineer.
🤖 Agent: Hi Alice! Nice to meet you—I'm here to help with any ML questions or challenges you have.

👤 User:  I'm working on a project about agent memory. Any tips?
🤖 Agent: Focus on a hybrid approach: combine episodic memory for specific past events with semantic memory for general knowledge learned over time.

👤 User:  What's my name and what do I do?
🤖 Agent: You're Alice, a machine-learning engineer working on agent memory.

缓冲区现在包含 6 条消息。
完成了 3 轮对话。

每轮 token 使用量：
  第  1 轮:    33 输入 token,   22 输出 token
  第  2 轮:    72 输入 token,   23 输出 token
  第  3 轮:   109 输入 token,   13 输出 token


检查原始消息缓冲区。这就是 LLM 每次调用接收到的全部内容。

In [ ]:
# Peek at the raw message buffer
for i, msg in enumerate(memory.get_history()):
    role_label = "USER" if msg["role"] == "user" else "ASST"
    # Truncate long messages for display
    preview = msg["content"][:80] + ("..." if len(msg["content"]) > 80 else "")
    print(f"  [{i}] {role_label}: {preview}")

## 线性 Token 增长问题

使用缓冲记忆时，**每条之前的消息在每轮都会被重新发送**。如下所示：

| 轮次 | 发送给 LLM 的 Token |
|------|---------------------|
| 1 | 仅第一条消息 |
| 2 | 消息 1-3 |
| 3 | 消息 1-5 |
| *n* | 消息 1-(2n-1) |

每轮输入 token 随对话长度**线性**增长。整个对话的**累计** token 使用量**二次方**增长。

In [20]:
print(f"完成了 {len(memory.turn_token_usage)} 轮对话。")
print("\n每轮 token 使用量：")
for t in memory.turn_token_usage:
    print(f"  第 {t['turn']:2d} 轮: {t['input_tokens']:5d} 输入 token, {t['output_tokens']:4d} 输出 token")

完成了 3 轮对话。

每轮 token 使用量：
  第  1 轮:    33 输入 token,   22 输出 token
  第  2 轮:    72 输入 token,   23 输出 token
  第  3 轮:   109 输入 token,   13 输出 token


## 持久化

生产环境的缓冲记忆必须能在进程重启后存活。数据结构是字典列表，所以 JSON 序列化（将其转换为文本文件格式）很直接。

In [22]:
import json

def save_conversation(memory: ConversationBufferMemory, path: str) -> None:
    """将消息缓冲区持久化为 JSON 文件。"""
    with open(path, "w") as f:
        json.dump(memory.get_history(), f, indent=2, ensure_ascii=False)
    print(f"已保存 {len(memory)} 条消息 → {path}")


def load_conversation(path: str, model, **kwargs) -> ConversationBufferMemory:
    """从 JSON 文件恢复 ConversationBufferMemory。"""
    with open(path) as f:
        messages = json.load(f)
    mem = ConversationBufferMemory(model=model, **kwargs)
    mem.messages = messages
    print(f"已加载 {len(messages)} 条消息 ← {path}")
    return mem


# 演示往返持久化
save_conversation(memory, "conversation.json")
loaded = load_conversation(
    "conversation.json",
    model=model,
    system_prompt="You are a helpful, concise assistant. Keep replies under 2 sentences.",
)

# 验证加载的记忆仍然可用
reply = loaded.chat("Remind me - what project was I working on?")
print(f"\n🤖 Agent（从加载的记忆）: {reply}")

已保存 6 条消息 → conversation.json
已加载 6 条消息 ← conversation.json

🤖 Agent（从加载的记忆）: You're working on a project about agent memory, specifically exploring how to implement it effectively.


## LangChain 对比

LangChain 提供了内置的 `ConversationBufferMemory`，做同样的事情。让我们看看框架版本与我们的 DIY 实现对比如何。

In [23]:
from langchain.memory import ConversationBufferMemory as LCBufferMemory
from langchain.chains import ConversationChain
from langchain.chat_models import init_chat_model

# 使用 DeepSeek 模型
lc_llm = init_chat_model(model="deepseek-chat")

lc_memory = LCBufferMemory(return_messages=True)

chain = ConversationChain(
    llm=lc_llm,
    memory=lc_memory,
    verbose=False,
)

# 同样的对话
print("👤:", "Hi, my name is Charlie.")
print("🤖:", chain.predict(input="Hi, my name is Charlie."), "\n")

print("👤:", "I'm building a knowledge graph for my company.")
print("🤖:", chain.predict(input="I'm building a knowledge graph for my company."), "\n")

print("👤:", "What's my name and what am I building?")
print("🤖:", chain.predict(input="What's my name and what am I building?"), "\n")

# 检查 LangChain 的内部缓冲区
print("--- LangChain 缓冲区内容 ---")
for msg in lc_memory.chat_memory.messages:
    role = msg.__class__.__name__.replace("Message", "")
    preview = msg.content[:80] + ("..." if len(msg.content) > 80 else "")
    print(f"  {role}: {preview}")

/var/folders/dv/lvctdkc52yxfd5kjhsc8m05h0000gn/T/ipykernel_50553/1393796113.py:8: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  lc_memory = LCBufferMemory(return_messages=True)
/var/folders/dv/lvctdkc52yxfd5kjhsc8m05h0000gn/T/ipykernel_50553/1393796113.py:10: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 1.0. Use :class:`~langchain_core.runnables.history.RunnableWithMessageHistory` instead.
  chain = ConversationChain(


👤: Hi, my name is Charlie.
🤖: Nice to meet you, Charlie! I’m glad you’re here. My name is—well, I don’t have a name in the traditional sense, but you can call me whatever fits the vibe of our conversation. Some people go with "Assistant" or "AI," and I’ve even been called "Buddy" or "Sage" before. 

So, Charlie—what’s on your mind today? Are you curious about something specific, looking for a bit of help, or just in the mood for a chat? I’ve got a whole library of facts, stories, and ideas tucked away, so feel free to steer us wherever you like. 

👤: I'm building a knowledge graph for my company.
🤖: That sounds like a fascinating project, Charlie! Knowledge graphs are incredibly powerful tools for organizing complex information and uncovering hidden relationships. I’ve seen them transform everything from search engines to recommendation systems.

Let’s dive into the details. To give you the most specific and helpful advice, I need to understand your use case a little better. There are 

**DIY vs. LangChain：并排对比：**

| 方面 | 我们的实现 | LangChain `ConversationBufferMemory` |
|------|-----------|--------------------------------------|
| **数据结构** | `list[dict]` 含 `role`/`content` | `ChatMessageHistory` 含类型化消息对象 |
| **API 集成** | 通过 LangChain `model.invoke()` 调用 | 封装在 `ConversationChain` 中 |
| **Token 跟踪** | 通过 `response.usage_metadata` 获取 | 通过回调可用 |
| **持久化** | DIY JSON 保存/加载 | 内置序列化选项 |
| **灵活性** | 完全控制提示词组装 | 固定的 chain 结构 |
| **代码行数** | ~50 行 | ~5 行（框架完成其余工作）|

**结论：** LangChain 适合原型开发。从零构建给你完全控制 token 预算、多模型路由和非标准消息格式。

## 权衡

### 适用场景
- **短对话**（20 轮以内），总 token 舒适地在上下文窗口内。
- **原型开发**：几分钟让事情跑起来，之后再优化。
- **需要完美回忆**：信息永远不会被丢弃或摘要。

### 失效场景
- **长对话**：每轮输入 token 线性增长。累计成本二次方增长。
- **成本敏感应用**：每次调用重新发送整个历史很昂贵。
- **延迟敏感应用**：更多输入 token 意味着更慢的响应。延迟是你的请求和模型回复之间的延迟。
- **上下文窗口限制**：最终缓冲区会超过模型的最大值（例如 Claude 的 200k token）。

### 成本示例
考虑一个 50 轮对话，每轮增加约 100 个 token 的新内容：
- 第 1 轮：约 100 个输入 token
- 第 50 轮：约 5,000 个输入 token
- **50 轮的总输入 token：约 127,500**（对比仅发送最新一轮的约 5,000）

### 下一步？
本系列后续内容解决这些限制：
- **[02：滑动窗口记忆](../02_sliding_window_memory/)** 只保留最近 *k* 条消息。
- **[03：摘要记忆](../03_summary_memory/)** 用 LLM 生成的摘要替换旧轮次。
- **[04：摘要缓冲记忆](../04_summary_buffer_memory/)** 是混合体：摘要旧消息，保留近期消息原文。
- **[05：Token 缓冲记忆](../05_token_buffer_memory/)** 按严格的 token 预算裁剪。

## 延伸阅读

- [Anthropic Messages API: Multi-turn Conversations](https://docs.anthropic.com/en/docs/build-with-claude/conversational-ai)
- [LangChain ConversationBufferMemory](https://python.langchain.com/docs/modules/memory/types/buffer)
- [LlamaIndex Chat Store](https://docs.llamaindex.ai/en/stable/module_guides/storing/chat_stores/)
- Anthropic's 7 Layers of Agent Memory (2026)
- [Lilian Weng, "LLM Powered Autonomous Agents" (Memory section)](https://lilianweng.github.io/posts/2023-06-23-agent/)

---

*下一篇：02：滑动窗口记忆 ->*